In [ ]:
import pandas as pd
import numpy as np

# =========================================
# LOAD FILE
# =========================================
file_path = "Sample of coupled data.xlsx"   # change path if needed

df = pd.read_excel(file_path, sheet_name="Sheet1", header=[0,1])

# =========================================
# FLATTEN MULTI-LEVEL HEADERS
# =========================================
df.columns = [' | '.join([str(c).strip() for c in col if pd.notna(c)])
              for col in df.columns]

# =========================================
# IDENTIFY PART / MATERIAL COLUMN
# =========================================
material_col = [col for col in df.columns if "Part" in col or "Material" in col][0]

materials = df[material_col]

# =========================================
# SELECT ONLY DEVIATION COLUMNS
# =========================================
deviation_cols = [col for col in df.columns if "Deviation" in col]

deviation_data = df[deviation_cols]

# =========================================
# ROW-WISE STATISTICS (same logic as yours)
# =========================================
mean_dev = deviation_data.mean(axis=1)
std_dev = deviation_data.std(axis=1, ddof=1)
count = deviation_data.count(axis=1)

std_error = std_dev / np.sqrt(count)

# 🔹 90% Confidence Interval
z = 1.645

lower_ci = mean_dev - z * std_error
upper_ci = mean_dev + z * std_error

# =========================================
# RESULT
# =========================================
result = pd.DataFrame({
    'Material': materials,
    'Mean_Deviation': mean_dev,
    'Std_Dev': std_dev,
    'Lower_90_CI': lower_ci,
    'Upper_90_CI': upper_ci
})

print(result)

# Optional save
result.to_excel("Deviation_Confidence_Output.xlsx", index=False)

In [ ]:
import pandas as pd
import numpy as np

file_path = "Coupled data Updated.xlsx"

df = pd.read_excel(file_path, sheet_name="Sheet2")

# Identify columns
part_col = "Part"
actual_cols = [col for col in df.columns if "Actual" in col]

# Extract
parts = df[part_col]
actual_data = df[actual_cols]

# Statistics
mean_actual = actual_data.mean(axis=1)
std_actual = actual_data.std(axis=1, ddof=1)

# 90% CI
z = 1.645

lower_actual = mean_actual - z * std_actual
upper_actual = mean_actual + z * std_actual

# Result
result = pd.DataFrame({
    "Part": parts,
    "Mean_Actual": mean_actual,
    "Std_Actual": std_actual,
    "Lower_Actual_90": lower_actual,
    "Upper_Actual_90": upper_actual
})

print(result)

result.to_excel("Actual_90CI_Output.xlsx", index=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================
# LOAD FILE
# ============================

file_path = "Coupled data Updated.xlsx"

df = pd.read_excel(file_path, sheet_name="Sheet2")

# ============================
# FIND ACTUAL COLUMNS
# ============================

actual_cols = [col for col in df.columns if "Actual" in col]

print("Actual columns detected:")
print(actual_cols)

# ============================
# SELECT PART (change index if needed)
# ============================

row_index = 0   # first part — change to analyze another part

values = df.loc[row_index, actual_cols].dropna()

print("\nValues used for histogram:")
print(values)

# ============================
# PLOT HISTOGRAM
# ============================

plt.figure(figsize=(8,5))
plt.hist(values, bins=10)
plt.title(f"Actual Distribution — Part: {df.loc[row_index, 'Part']}")
plt.xlabel("Actual Values")
plt.ylabel("Frequency")
plt.show()